# Занятие 3, пара 2. Парсинг. Чистка, пагинация, законность


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


Первую страницу мы уже умеем читать. Великолепно! Но одна страница — ещё не весь каталог, а надпись «По запросу» — не цена в ноль рублей. Давайте доведём сбор до результата, которому можно хотя бы задать несколько вопросов.

Сегодня понадобятся циклы, функции и pandas. Итог — таблица, которую можно прочитать заново, и короткая запись о том, откуда она взялась.


### Подготовка

Работайте в своей копии внутри папки проекта. Запустите ячейки по порядку. Эта тетрадь не использует переменные из другой тетради. GitHub — по желанию; сохраняйте файл и отдельную копию.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


In [ ]:
import re
import json
import pandas as pd
from urllib.robotparser import RobotFileParser
from IPython.display import display
from course_support.parsing import parse_catalog_stdlib
try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None
    print('Используем резервный разбор учебного каталога.')
CATALOG_DIR = DATA_DIR / 'parsing'
OUTPUT_DIR = WORKSPACE_DIR / 'reports' / 'parsing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_html(name):
    # Учебные ссылки указывают только на HTML в этой папке.
    if Path(name).name != name or not name.endswith('.html'):
        raise ValueError('Ожидалось имя локальной HTML-страницы')
    return (CATALOG_DIR / name).read_text(encoding='utf-8')

def parse_page(html):
    if BeautifulSoup is None:
        return parse_catalog_stdlib(html)
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for card in soup.select('article.listing'):
        row = {'listing_id': card['data-id']}
        for field in ['title', 'address', 'price', 'area']:
            row[field] = card.select_one(f'[data-field="{field}"]').get_text(' ', strip=True)
        row['href'] = card.select_one('[data-field="title"]')['href']
        rows.append(row)
    next_link = soup.select_one('a.next')
    return rows, next_link['href'] if next_link else None

rows, next_name = parse_page(load_html('page-1.html'))
display(pd.DataFrame(rows))
print('Следующая:', next_name)


## Можно получить — не значит можно использовать как угодно
Перед сбором реальных данных смотрим условия источника и разрешённые способы доступа, назначение данных и ограничения публикации. Для проекта часто проще взять разрешённый CSV или API. Не собираем лишние контакты и другие персональные сведения.

`robots.txt` — правила для автоматических клиентов. Его результат **не является разрешением на доступ или использование данных**. Сейчас проверим только правило для двух путей на вымышленном сайте: никакого запроса туда нет. Для нашей локальной синтетики учебное использование описано в SOURCE.md.


In [ ]:
robots = RobotFileParser()
robots.parse((CATALOG_DIR / 'robots.txt').read_text(encoding='utf-8').splitlines())
print((CATALOG_DIR / 'robots.txt').read_text(encoding='utf-8'))


### ✏️ Ваш ход 1: прочитать правило

С помощью `robots.can_fetch('UniversityPractice', адрес)` получите `allowed_catalog` для `https://example.invalid/catalog/` и `allowed_private` для `https://example.invalid/private/`. Объясните, почему первое True само по себе ничего не говорит о лицензии данных.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.2.1', allowed_catalog, allowed_private)


## Пробелы бывают разные
Для лишних пробелов в адресе пригодится знакомая связка `split` и `join`: сначала разделить на слова, потом соединить одним пробелом. `split()` без аргумента понимает и обычный, и неразрывный пробел. А вот `split(' ')` ведёт себя иначе.

Цену будем разбирать строго в формате этого набора: целые рубли с пробелами и знаком ₽ либо «По запросу». Если встретится другая запись, лучше увидеть ошибку и проверить её, чем незаметно превратить «3–5 млн» в 35.


In [ ]:
address = '  Примерная   улица, 2\n'
print(repr(address), '→', ' '.join(address.split()))
price_text = '4\u00a0200\u202f000 ₽'
print(repr(price_text))
compact = ''.join(price_text.split())
print(compact, re.fullmatch(r'(\d+)₽', compact).group(1))


### ✏️ Ваш ход 2: число или пропуск

Напишите `parse_price(text)`: «По запросу» → `None`; целые рубли со знаком ₽ → `int`; незнакомый формат → `ValueError`. Уберите пробелы через `''.join(text.split())`; для числа используйте `re.fullmatch`. Знак валюты проверьте, а не удаляйте все нецифровые символы подряд.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.2.2', parse_price)


## Пагинация: идём по «Следующей»
Номер последней страницы заранее не нужен. У каждой страницы спрашиваем ссылку на следующую; если её нет, заканчиваем.

Пригодятся ещё две страховки: множество уже посещённых страниц и предел числа шагов. Если ссылка зациклилась или лимит достигнут, **сообщаем об ошибке**, а не выдаём кусок данных за весь каталог. Один ID может встретиться на двух страницах: это не причина прекращать обход — дальше могут быть новые карточки.


In [ ]:
def collect_pages(start='page-1.html', loader=load_html, max_pages=10):
    name = start
    visited = []
    records = []
    while name is not None:
        if name in visited:
            raise ValueError(f'Ссылка зациклилась: {name}')
        if len(visited) >= max_pages:
            raise ValueError('Достигнут предел страниц; полнота не подтверждена')
        rows, next_name = parse_page(loader(name))
        if not rows:
            raise ValueError(f'Нет карточек: проверьте разметку {name}')
        for row in rows:
            records.append({**row, 'source_page': name})
        visited.append(name)
        name = next_name
    return pd.DataFrame(records), visited

raw_df, visited = collect_pages()
print('Страницы:', visited)
print('Карточек:', len(raw_df), 'уникальных ID:', raw_df['listing_id'].nunique())
display(raw_df[raw_df.duplicated('listing_id', keep=False)])


### ✏️ Ваш ход 3: пригодная таблица

Создайте `clean_df`: оставьте одну запись на `listing_id`, нормализуйте пробелы адреса, добавьте `price_rub` через свою функцию. Используйте тип pandas `Int64` с большой I — он допускает пропуски в целых числах. Исходные `price`, `source_page` и остальные поля сохраните.
Посчитайте строки и пропуски. В Markdown объясните, почему отсутствие цены и нулевая цена не одно и то же.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.2.3', clean_df)


## Сохраним и прочитаем заново
CSV не хранит типы столбцов так же, как DataFrame. Поэтому при обратном чтении явно укажем ID-строку и целую цену с пропусками. Вместе с выгрузкой оставим происхождение: другому человеку не придётся гадать, откуда взялись цифры.


In [ ]:
csv_path = OUTPUT_DIR / 'listings.csv'
clean_df.to_csv(csv_path, index=False, encoding='utf-8')
restored = pd.read_csv(csv_path, dtype={'listing_id': 'string', 'price_rub': 'Int64'})
assert len(restored) == 10 and restored['price_rub'].isna().sum() == 2
source_info = {
    'source': 'Учебный синтетический каталог курса А. А. Дуркина',
    'files': visited, 'raw_rows': len(raw_df), 'unique_rows': len(clean_df),
    'key': 'listing_id', 'price_unit': 'RUB',
    'missing_price': 'Цена не указана; не ноль',
    'limitations': 'Вымышленный набор, не оценка реального рынка',
    'usage': 'Учебное использование с указанием источника; см. SOURCE.md',
}
(OUTPUT_DIR / 'source.json').write_text(json.dumps(source_info, ensure_ascii=False, indent=2), encoding='utf-8')
print(csv_path)
print('Средняя среди известных цен:', clean_df['price_rub'].mean())


## Практический запас — 15–20 минут
**Опыт 1.** Сравните среднюю известных цен и среднюю после `fillna(0)`. Какое предположение о квартирах вы незаметно добавили во втором случае?

**Опыт 2.** Устроим зацикливание только в памяти. Предскажите, чем закончится следующий запуск. После опыта исходные файлы остаются прежними.


In [ ]:
def loop_loader(name):
    text = load_html(name)
    if name == 'page-3.html':
        text = text.replace('</nav>', '<a class="next" href="page-1.html">Назад</a></nav>')
    return text

try:
    collect_pages(loader=loop_loader)
except ValueError as error:
    print('Ожидаемая остановка:', error)


**Опыт 3.** Для своей темы (или временного каталога) запишите: источник, единицу наблюдения, 3 нужных поля, способ получения, известные ограничения. Не надо срочно парсить реальный сайт или регистрироваться где-либо.

## Итог
В своей тетради допишите: сколько карточек получили, сколько осталось после удаления повторов, как поступили с пропусками и какой вывод по этой таблице делать нельзя. Сохраните тетрадь, listings.csv и source.json. ЛР1 продолжает действовать по своим критериям; эта практика готовит к сбору данных в ЛР2 после блока API. Новый срок преподаватель объявит отдельно.

Справка: [протокол robots.txt](https://www.rfc-editor.org/rfc/rfc9309.html), [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).
